# COVID Task-Transfer Matrix

Fixed-adaptation transfer results for COVID-only train-task to eval-task experiments. Rows are source/pretraining tasks, columns are eval tasks after 1000 adaptation steps.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("task_matrix_adapt_1000.csv")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

row_order = ["scratch", "nm", "cl", "fp"]
col_order = ["nm", "cl", "fp"]
labels = {"scratch": "Scratch", "nm": "NM", "cl": "CL", "fp": "FP"}

df = pd.read_csv(DATA)
df["value"] = pd.to_numeric(df["value"])
df.head()

In [ ]:
def matrix(metric):
    return (
        df[df["metric"].eq(metric)]
        .pivot(index="train_task", columns="eval_task", values="value")
        .reindex(index=row_order, columns=col_order)
    )

primary = matrix("primary")
auc = matrix("roc_auc")
accuracy = matrix("accuracy")
loss = matrix("loss")
score = matrix("score")

primary

The primary matrix mixes metrics: NM/CL use ROC-AUC, while FP uses negative MSE. The plots below treat each eval task separately, zoom the y-axis around the observed range, and compare each pretraining row against scratch.

In [ ]:
colors = {"scratch": "#8a8f98", "nm": "#2f6f9f", "cl": "#5a9f72", "fp": "#b45f4d"}


def style_axis(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#d7d7d7", linewidth=0.8, alpha=0.7)
    ax.set_axisbelow(True)


def annotate_bars(ax, bars, fmt):
    ymin, ymax = ax.get_ylim()
    pad = (ymax - ymin) * 0.025
    for bar in bars:
        value = bar.get_height()
        va = "bottom" if value >= 0 else "top"
        offset = pad if value >= 0 else -pad
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + offset,
            fmt(value),
            ha="center",
            va=va,
            fontsize=9,
        )


def bar_by_source(values, title, ylabel, filename, baseline=None, ylim=None, fmt=lambda x: f"{x:.4f}"):
    values = values.reindex(row_order)
    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    bars = ax.bar(
        [labels[x] for x in values.index],
        values.values,
        color=[colors[x] for x in values.index],
        width=0.68,
    )
    if baseline is not None:
        ax.axhline(baseline, color="#3a3a3a", linestyle="--", linewidth=1.2, label="Scratch")
        ax.legend(frameon=False, loc="best")
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    style_axis(ax)
    annotate_bars(ax, bars, fmt)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=200)
    return fig, ax


def padded_limits(values, pad_frac=0.25, min_pad=1e-4):
    values = values.dropna()
    lo, hi = values.min(), values.max()
    pad = max((hi - lo) * pad_frac, min_pad)
    return lo - pad, hi + pad


bar_by_source(
    auc["nm"],
    "Eval: NM after 1000 adaptation steps",
    "ROC-AUC",
    "nm_auc_by_pretrain.png",
    baseline=auc.loc["scratch", "nm"],
    ylim=padded_limits(auc["nm"], pad_frac=0.18, min_pad=0.002),
    fmt=lambda x: f"{x:.4f}",
);
bar_by_source(
    auc["cl"],
    "Eval: CL after 1000 adaptation steps",
    "ROC-AUC",
    "cl_auc_by_pretrain.png",
    baseline=auc.loc["scratch", "cl"],
    ylim=padded_limits(auc["cl"], pad_frac=0.35, min_pad=0.000025),
    fmt=lambda x: f"{x:.6f}",
);
bar_by_source(
    loss["fp"] * 1e4,
    "Eval: FP after 1000 adaptation steps",
    "MSE loss x 1e4 (lower is better)",
    "fp_loss_by_pretrain.png",
    baseline=loss.loc["scratch", "fp"] * 1e4,
    ylim=padded_limits(loss["fp"] * 1e4, pad_frac=0.22, min_pad=0.02),
    fmt=lambda x: f"{x:.3f}",
);


In [ ]:
improvement = pd.DataFrame(index=row_order, columns=col_order, dtype=float)
for eval_task in ["nm", "cl"]:
    improvement[eval_task] = auc[eval_task] - auc.loc["scratch", eval_task]

# FP uses loss, so positive improvement means lower loss than scratch.
improvement["fp"] = loss.loc["scratch", "fp"] - loss["fp"]
improvement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.4), gridspec_kw={"width_ratios": [1.35, 1.0]})

auc_delta = improvement[["nm", "cl"]].rename(index=labels, columns={"nm": "NM AUC", "cl": "CL AUC"})
auc_delta.plot(kind="bar", ax=axes[0], width=0.74, color=["#2f6f9f", "#5a9f72"])
axes[0].axhline(0, color="#3a3a3a", linewidth=1.0)
axes[0].set_title("AUC improvement over scratch")
axes[0].set_ylabel("ROC-AUC delta")
axes[0].set_xlabel("")
axes[0].legend(frameon=False, title="Eval task")
style_axis(axes[0])
for container in axes[0].containers:
    annotate_bars(axes[0], container, lambda x: f"{x:+.4f}")

fp_delta = (improvement["fp"] * 1e6).rename(index=labels)
fp_bars = axes[1].bar(
    fp_delta.index,
    fp_delta.values,
    color=[colors[k] for k in row_order],
    width=0.68,
)
axes[1].axhline(0, color="#3a3a3a", linewidth=1.0)
axes[1].set_title("FP loss reduction over scratch")
axes[1].set_ylabel("MSE loss reduction x 1e6")
axes[1].set_xlabel("")
style_axis(axes[1])
annotate_bars(axes[1], fp_bars, lambda x: f"{x:+.1f}")

fig.suptitle("Fixed-adaptation transfer after 1000 steps", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "improvement_vs_scratch.png", dpi=200, bbox_inches="tight")

pd.concat(
    [auc_delta, fp_delta.rename("FP loss reduction x1e6")],
    axis=1,
)


In [ ]:
summary_rows = []
for eval_task in col_order:
    if eval_task in ["nm", "cl"]:
        metric_values = auc[eval_task]
        best_source = metric_values.idxmax()
        scratch_value = metric_values.loc["scratch"]
        summary_rows.append({
            "eval_task": eval_task.upper(),
            "metric": "ROC-AUC",
            "best_source": labels[best_source],
            "best_value": metric_values.loc[best_source],
            "scratch_value": scratch_value,
            "delta_vs_scratch": metric_values.loc[best_source] - scratch_value,
        })
    else:
        metric_values = loss[eval_task]
        best_source = metric_values.idxmin()
        scratch_value = metric_values.loc["scratch"]
        summary_rows.append({
            "eval_task": eval_task.upper(),
            "metric": "MSE loss",
            "best_source": labels[best_source],
            "best_value": metric_values.loc[best_source],
            "scratch_value": scratch_value,
            "delta_vs_scratch": scratch_value - metric_values.loc[best_source],
        })

summary = pd.DataFrame(summary_rows)
summary

## Matrix View

The raw matrix mixes metric families, so it is shown as a table instead of a shared-color heatmap. NM/CL cells are ROC-AUC; FP cells are MSE loss scaled by `1e4` where lower is better. The cross-task plot excludes scratch and same-task diagonal cells, then compares each source task to the scratch baseline for the eval task.

In [ ]:
matrix_rows = ["scratch", "nm", "cl", "fp"]
matrix_cols = ["nm", "cl", "fp"]
source_colors = {"scratch": "#8a8f98", "nm": "#2f6f9f", "cl": "#5a9f72", "fp": "#b45f4d"}

# Raw matrix as table: do not use a shared colorbar across AUC and loss.
table_values = []
best_cells = set()
for eval_task in matrix_cols:
    if eval_task in ["nm", "cl"]:
        best_train = auc[eval_task].idxmax()
    else:
        best_train = loss[eval_task].idxmin()
    best_cells.add((best_train, eval_task))

for train in matrix_rows:
    row = []
    for eval_task in matrix_cols:
        if eval_task in ["nm", "cl"]:
            row.append(f"{auc.loc[train, eval_task]:.4f}")
        else:
            row.append(f"{loss.loc[train, eval_task] * 1e4:.3f}")
    table_values.append(row)

fig_table, ax = plt.subplots(figsize=(7.4, 3.5))
ax.axis("off")
table = ax.table(
    cellText=table_values,
    rowLabels=[labels[r] for r in matrix_rows],
    colLabels=["NM\nROC-AUC ↑", "CL\nROC-AUC ↑", "FP\nMSE x1e4 ↓"],
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.18, 1.85)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#d0d0d0")
    cell.set_linewidth(0.8)
    if row == 0 or col == -1:
        cell.set_facecolor("#f1f3f5")
        cell.set_text_props(weight="bold")
    if row > 0 and col >= 0:
        train = matrix_rows[row - 1]
        eval_task = matrix_cols[col]
        if (train, eval_task) in best_cells:
            cell.set_facecolor("#dff0d8")
            cell.set_text_props(weight="bold")
        elif train == "scratch":
            cell.set_facecolor("#f7f7f7")

ax.set_title("Fixed-adaptation task-transfer matrix", pad=18)
ax.text(
    0.5,
    -0.08,
    "Green cells are best within each eval task column. FP is lower-is-better loss.",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=10,
)
fig_table.tight_layout()
fig_table.savefig(FIG_DIR / "task_transfer_matrix_table.png", dpi=200, bbox_inches="tight")

# Cross-task-only deltas vs scratch. Diagonal/self-transfer cells are intentionally excluded.
auc_rows = []
for eval_task in ["nm", "cl"]:
    for train in ["nm", "cl", "fp"]:
        if train == eval_task:
            continue
        auc_rows.append({
            "label": f"{labels[train]}→{eval_task.upper()}",
            "train": train,
            "eval_task": eval_task,
            "delta": auc.loc[train, eval_task] - auc.loc["scratch", eval_task],
        })

fp_rows = []
for train in ["nm", "cl"]:
    fp_rows.append({
        "label": f"{labels[train]}→FP",
        "train": train,
        "delta": (loss.loc["scratch", "fp"] - loss.loc[train, "fp"]) * 1e6,
    })

fig_delta, axes = plt.subplots(1, 2, figsize=(10.8, 4.0), gridspec_kw={"width_ratios": [1.4, 0.9]})

auc_values = [r["delta"] for r in auc_rows]
auc_bars = axes[0].bar(
    [r["label"] for r in auc_rows],
    auc_values,
    color=[source_colors[r["train"]] for r in auc_rows],
    width=0.68,
)
auc_pad = max((max(auc_values) - min(auc_values)) * 0.08, 0.00045)
axes[0].set_ylim(min(min(auc_values) - auc_pad, -auc_pad), max(auc_values) + auc_pad * 2.2)
axes[0].axhline(0, color="#333333", linewidth=1.0)
axes[0].set_title("Cross-task AUC delta vs scratch")
axes[0].set_ylabel("ROC-AUC delta")
axes[0].tick_params(axis="x", rotation=25)
style_axis(axes[0])
for bar in auc_bars:
    value = bar.get_height()
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        value + (auc_pad * 0.35 if value >= 0 else -auc_pad * 0.35),
        f"{value:+.4f}",
        ha="center",
        va="bottom" if value >= 0 else "top",
        fontsize=9,
    )

fp_values = [r["delta"] for r in fp_rows]
fp_bars = axes[1].bar(
    [r["label"] for r in fp_rows],
    fp_values,
    color=[source_colors[r["train"]] for r in fp_rows],
    width=0.58,
)
fp_pad = max((max(fp_values) - min(fp_values)) * 0.12, 1.8)
axes[1].set_ylim(min(fp_values) - fp_pad, max(max(fp_values) + fp_pad, fp_pad))
axes[1].axhline(0, color="#333333", linewidth=1.0)
axes[1].set_title("Cross-task FP delta vs scratch")
axes[1].set_ylabel("MSE loss reduction x1e6")
axes[1].tick_params(axis="x", rotation=25)
style_axis(axes[1])
for bar in fp_bars:
    value = bar.get_height()
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        value + (fp_pad * 0.28 if value >= 0 else -fp_pad * 0.28),
        f"{value:+.1f}",
        ha="center",
        va="bottom" if value >= 0 else "top",
        fontsize=9,
    )

fig_delta.suptitle("Cross-task transfer after 1000 adaptation steps", y=1.02)
fig_delta.tight_layout()
fig_delta.savefig(FIG_DIR / "cross_task_delta_panels.png", dpi=200, bbox_inches="tight")

# Keep the old filename updated for convenience when viewing the notebook outputs.
fig_delta.savefig(FIG_DIR / "task_transfer_matrices.png", dpi=200, bbox_inches="tight")

pd.DataFrame(auc_rows), pd.DataFrame(fp_rows)

